In [ ]:
import pandas as pd
import os
from pathlib import Path
import sys

# === Input configuration ===
table_path = Path("res_init2table.tsv")   # Atom metadata table
metal_mask = ":371@MG"                         # Fixed metal atom selection (e.g., Mg)

# === Load the TSV file into a MultiIndexed DataFrame ===
df = pd.read_csv(table_path, sep='\t', index_col=[0, 1, 2, 3])
idx = df.index  # MultiIndex: category, resname, resid, atom_name


# === Define filters for specific coordination logic ===

# 1. Keep all entries from 'coord' category with oxygen atoms, excluding 'nuc', 'base', and 'O' atom in rescoord
is_coord = idx.get_level_values("category").str.contains("coord", case=False)
is_oxygen = idx.get_level_values("atom_name").str.startswith("O")
is_not_nuc = ~idx.get_level_values("category").str.contains("nuc", case=False)
is_not_base = ~idx.get_level_values("category").str.contains("base", case=False)
is_not_rescoord_O = ~(
    (idx.get_level_values("category") == "rescoord") &
    (idx.get_level_values("atom_name") == "O")
)
coord_filter = is_coord & is_oxygen & is_not_nuc & is_not_base & is_not_rescoord_O

# 2. Keep only "O3'" from dna3term
dna3_filter = (
    idx.get_level_values("category") == "dna3term"
) & (
    idx.get_level_values("atom_name") == "O3'"
)

# 3. Keep only "OP2" from dna5term
dna5_filter = (
    idx.get_level_values("category") == "dna5term"
) & (
    idx.get_level_values("atom_name") == "OP2"
)

# === Combine all valid filters ===
final_mask = coord_filter | dna3_filter | dna5_filter
filtered = df[final_mask]

# === Save AMBER mask output to a TSV file named after this script ===
def save_amber_mask(dataframe):
    try:
        script_name = os.path.splitext(os.path.basename(__file__))[0]
    except NameError:
        script_name = "get_coordination"
    output_filename = f"{script_name}.tsv"

    lines = [f"{metal_mask} :{resid}@{atom_name}" for _, _, resid, atom_name in dataframe.index]
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("Atoms Pairs\n")
        f.write("\n".join(lines))

save_amber_mask(filtered)